# TP 4: ORIENTATION DETECTION VIA NEUROGEOMETRIC APPROACH 
## Contact: emre.baspinar@inria.fr

Emre Baspinar, Inria Branch of the University of Montpellier, MathNeuro Team

## Initialization

In [ ]:
#################################################################################################################
## Initialization ###############################################################################################
#################################################################################################################

import numpy as np
import cv2
from PIL import Image
from scipy.ndimage import gaussian_filter
from matplotlib import pyplot as plt
from gaborTransform import gaborFunction, gaborFilterBank, gaborTransform, inverseGaborTransform
import sys
# !{sys.executable} -m pip install mpl-interactions
# !{sys.executable} -m pip install ipympl

# !pip install mpl-interactions
# !pip install ipympl
# !pip install mpl_interactions[jupyter]
# !pip install opencv-python

## Construct the Gabor function

## Step 1
a) Observe how "sigma" affects the Gabor function by varying it. What does scale determine in the Gabor functions?

b) Change the wavelength "Lambda" by varying "param2" and observe its effects on the Gabor function. What is the difference between the effects of changing the wavelength and of changing the scale?

c) So far, we have considered only the even component of the Gabor functions. What is the difference between the even and odd components of the Gabor functions? Change the phase "psi" to $\pi/2$ and observe the resultant Gabor function. Which component is the resultant function? How the even and odd components of the Gabor function are related to each other via the phase "psi"?

d) Change the ellipticity ratio "gamma" and observe the results. What does the ellipticity ratio determine?

e) Change the number "nOfTheta" of the orientation samples and observe the effects. What does the parameter "nOfTheta" determine?


In [ ]:
# Gabor filter parameters
sigma  = 4            # scale 
# Lambda = 2 * sigma    # wavelength
psi    = 0            # phase offset
gamma  = 0.8          # ellipticity ratio
# b      = 1          # half-response spatial frequency bandwidth (in octaves)
periodicity = np.pi   # peridicity of the orientation angles
nOfTheta = 16         # number of orientation samples between 0 and periodicity


# Construct & visualize a single Gabor function
# %matplotlib inline 
# plt.imshow(gaborFunction(theta, 8, psi, sigma, gamma), interpolation='none', cmap='gray') # if sigma is not 0, b is not used.
# plt.xlabel('x axis')
# plt.ylabel('y axis')
# plt.show()
%matplotlib ipympl

from mpl_interactions import ipyplot as iplt

def f(param1, param2):
    return gaborFunction(param1, param2, psi, sigma, gamma)

# param1 is the orientation parameter \theta of the Gabor function
# param2 is the wavelength

fig, ax = plt.subplots()
controls = iplt.imshow(f, param1=(0, np.pi-np.pi/nOfTheta), param2=(2*np.pi, 4*np.pi), interpolation='none', cmap='gray',autoscale_cmap=True)


## Generate the test image

## Step 2
a) Change "thickness" and the radius "r". Observe what they change in the test image.

b) What does smoothing do on the test image? What is possible advantages of smoothing?

In [ ]:
##### Test image is a circle with a certain thickness...

# Test image parameters
imSize     = 128       # size of the image (in pixels)
a = b = imSize // 2    # x and y components of the position of the origin of the circle in the test image
sigmaGauss = 4         # scale of the Gaussian for initial smoothing

# Circle parameters
r = 40                 # radius
# thickness = 4          # ring thickness (in pixels)
thickness = 4          # ring thickness (in pixels)

# Coordinate grid
y, x = np.ogrid[:imSize, :imSize]

# Radial distance from center
dist = np.sqrt((x - a)**2 + (y - b)**2)

# Create ring
imArray = np.logical_and(
    dist >= r - thickness/2,
    dist <= r + thickness/2
).astype(float)

# Optional smoothing
imSmooth = gaussian_filter(imArray, sigma=2)

# Display
plt.figure(figsize=(8,4))

plt.subplot(1,2,1)
plt.title(" Ring")
plt.imshow(imArray, cmap="gray")
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Smoothed ring")
plt.imshow(imSmooth, cmap="gray")
plt.axis("off")

plt.show()

## Construct the Gabor filter bank
# Step 3
Decrease nOfTheta and observe the resultant Gabor filter bank. What does nOfTheta determine in the Gabor filter bank?


In [ ]:
# Gabor filter parameters
sigma  = 4            # scale 
Lambda = 2 * sigma    # wavelength
psi    = 0            # phase offset
gamma  = 0.8          # ellipticity ratio
# b      = 1          # half-response spatial frequency bandwidth (in octaves)
periodicity = np.pi   # peridicity of the orientation angles
nOfTheta = 16         # number of orientation samples between 0 and periodicity


# Construct & visualize a Gabor filter bank with \pi-periodicity
filterBank = gaborFilterBank(Lambda, psi, sigma, nOfTheta, gamma)

fig = plt.figure(figsize= (8, 8))
for i in range(0,nOfTheta):
    # required nrows=4, required ncoms=4, index_location= i+1
    ax = fig.add_subplot(4, 4, i+1)
    # x_batch[i]: Image object at each iteration
    ax.imshow(filterBank[i], cmap='gray')

## Gabor transform on the test image: lifting the image to the cortical space
# Step 4
Run the Gabor transform with nOfTheta = 8 and with nOfTheta = 16. What do you observe? Why is it important to choose nOfTheta sufficiently high?

In [ ]:
liftedArray = np.empty([nOfTheta,imSize, imSize], dtype=float) # initialize the lifted array

fig = plt.figure(figsize= (8, 8))
for i in range(0, nOfTheta):    
    kernel = filterBank[i]
    out = cv2.filter2D(imSmooth, ddepth= -1, kernel=kernel)
    liftedArray[i] = out
    # required nrows=4, required ncoms=4, index_location= i+1
    ax = fig.add_subplot(4, 4, i+1)
    # x_batch[i]: Image object at each iteration  
    ax.imshow(out, cmap='gray')

## Inverse Gabor transform (degenerate) on the lifted image (for one single frequency): projecting the output responses from cortical space to the image plane
# Step 5
What does (degenerate) inverse transform provide? From which space to which space does this inverse transform map the output responses of the simple cells?

In [ ]:
invArray = inverseGaborTransform(liftedArray, Lambda, psi, sigma, gamma=1, periodicity = np.pi)
fig = plt.figure(figsize= (8, 8))
plt.imshow(invArray, interpolation='none', cmap='gray')
plt.show()